# Module 4: Q1, where are prices rising fastest?

**Business question:** which Dubai communities saw the fastest growth in price per sq ft, for which unit types, and is that growth still going in 2026?

**Input:** `agg_price_community_yearly` and the star schema from Module 3
**Output:** three results tables (`q1_yoy`, `q1_growth_2021_2025`, `q1_momentum_2026`), their CSV exports, and your **written findings** for the README

**Time:** about 2 hours. This module is almost all **SQL**, which is the skill most analyst interviews test hardest.

**New skills:** window functions (`LAG`, `RANK`, `DENSE_RANK`, `ROW_NUMBER`), `PARTITION BY`, the `WINDOW` and `QUALIFY` clauses, temporary views, CTEs, pivoting with conditional aggregation, CAGR, like-for-like comparisons, minimum-sample rules, and writing up findings.

## 🧠 Why we're doing this: GROUP BY vs window functions

`GROUP BY` **collapses** rows: 10 rows in, 1 row out. A **window function** calculates *across related rows* but **keeps every row**. That's exactly what "compare this year with last year" needs: each row stays, and you add the previous year's value next to it.

```
FUNCTION() OVER (PARTITION BY ...   -- which rows belong together (like GROUP BY, but without collapsing)
                 ORDER BY ...)      -- the order inside each group (needed for "previous" and "rank")
```

**You already know this from Excel:** in a sheet sorted by community and year, `=IF(A3=A2, C3/C2-1, "")` compares a row with the one above it *only if it's the same community*. `PARTITION BY` is the "same community" test, `ORDER BY` is the sorting, and `LAG()` is "the row above".

| Function | Returns | Typical use |
|----------|---------|-------------|
| `LAG(x)` / `LEAD(x)` | value from the previous / next row | year-on-year change |
| `ROW_NUMBER()` | 1, 2, 3, 4 (always unique) | pick exactly one row per group |
| `RANK()` | 1, 2, 2, 4 (ties share, then skip) | league tables |
| `DENSE_RANK()` | 1, 2, 2, 3 (ties share, no gaps) | "top 3 distinct values" |
| `SUM(x) OVER (...)` | running or group total on every row | share of total |

🎤 **Interview angle:** window functions are the most-asked SQL topic in analyst interviews. After this module you'll have a real project example for each one.

## Setup

In [ ]:
from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "dubai_re"
T = lambda name: f"{CATALOG}.{SCHEMA}.{name}"

MIN_SALES     = 50   # minimum sales in EACH year being compared (see Step 1)
MIN_SHIFT_PTS = 25   # off-plan share change (percentage points) that flags a mix shift (see Step 5c)

def check(label, actual, expected):
    fmt = (lambda v: f"{v:,}") if isinstance(expected, int) else str
    if actual == expected:
        print(f"✅ {label}: {fmt(actual)}")
    else:
        print(f"❌ {label}: got {actual}, expected {fmt(expected)}. Re-read the hint above this cell.")

def check_close(label, actual, expected, tolerance=0.03):
    if actual is None:
        print(f"❌ {label}: got nothing back (None). Check the cell above.")
        return
    ok = abs(actual - expected) <= tolerance * abs(expected)
    f = ",.0f" if isinstance(expected, int) else ",.1f"
    print(f"{'✅' if ok else '❌'} {label}: got {actual:{f}}, expected about {expected:{f}} (±{tolerance:.0%})")

## Step 1: How much evidence is behind each median?

🧠 **Why we're doing this**
`agg_price_community_yearly` has a median price for every year × community × segment × bedroom band, but some of those medians rest on **one or two sales**. A community "rising 80%" because two flats sold at different prices isn't a finding. It's noise. Good analysts check sample sizes **before** ranking anything.

This query buckets the 2025 rows by how many sales sit behind each median. Run it and look at two things: how many *rows* are in the small buckets, and how much of the *market* (sales) they represent.

`SUM(...) OVER ()`, a window over **all** rows, puts the grand total on every row, so each bucket can show its share. That's your first window function.

In [ ]:
%sql
WITH buckets AS (
  SELECT
    CASE WHEN sales < 10  THEN '1. fewer than 10'
         WHEN sales < 50  THEN '2. 10 to 49'
         WHEN sales < 200 THEN '3. 50 to 199'
         ELSE                  '4. 200 or more' END AS sales_bucket,
    sales
  FROM workspace.dubai_re.agg_price_community_yearly
  WHERE year = 2025
)
SELECT sales_bucket,
       COUNT(*)                                                 AS groups,
       SUM(sales)                                               AS sales,
       ROUND(100.0 * SUM(sales) / SUM(SUM(sales)) OVER (), 1)   AS pct_of_2025_sales
FROM buckets
GROUP BY sales_bucket
ORDER BY sales_bucket

**What you should see:** roughly **55% of the 2025 groups have fewer than 50 sales**, yet together they account for only about **3% of the sales**. A minimum of 50 sales a year (`MIN_SALES`) drops the noisy groups and keeps the vast majority of the market. It's a judgement call, and that's fine as long as it's written down and justified. You'll mention it in your README.

`SUM(SUM(sales)) OVER ()` looks odd. The inner `SUM` is the normal `GROUP BY` total per bucket. The outer `SUM … OVER ()` then adds up those bucket totals across all rows. Window functions run *after* `GROUP BY`, which is why they can wrap an aggregate.

## Step 2: `LAG()`, putting last year next to this year

🧠 **Why we're doing this**
Start small: one series, JVC 1-bed apartments. `LAG(median_aed_sqft) OVER (PARTITION BY … ORDER BY year)` returns the previous year's median *for the same community, segment and bedroom band*.

The **`WINDOW w AS (...)`** clause at the bottom names the window once, so three `LAG`s don't repeat the same `PARTITION BY … ORDER BY …`. Shorter, and impossible to get inconsistent.

✏️ **TODO 2:** add `LAG(year) OVER w AS prev_year` and `LAG(median_aed_sqft) OVER w AS prev_median` (the first `LAG` is done for you).

In [ ]:
%sql
-- TODO 2: add prev_year and prev_median
SELECT year, sales, median_aed_sqft,
       LAG(sales) OVER w AS prev_sales,
       ___,
       ___
FROM workspace.dubai_re.agg_price_community_yearly
WHERE community = 'Jumeirah Village Circle' AND segment = 'Apartment' AND bedroom_band = '1 BR'
WINDOW w AS (PARTITION BY community, segment, bedroom_band ORDER BY year)
ORDER BY year

Check the 2015 row: all three `prev_` columns are NULL, because there *is* no row above it in that partition. That's correct, not an error.

## Step 3: The gap trap, then the full year-on-year table

🧠 **Why we're doing this**
`LAG` returns the **previous row**, not the **previous year**. If a community had sales in 2019, none in 2020, and sales again in 2021, then for 2021 `LAG` hands you **2019**. You'd label a two-year change as "year-on-year" without noticing.

Run the cell below. It counts how often that happens in this table.

In [ ]:
gaps = spark.sql(f"""
    WITH lagged AS (
        SELECT year, LAG(year) OVER (PARTITION BY community, segment, bedroom_band ORDER BY year) AS prev_year
        FROM {T("agg_price_community_yearly")}
    )
    SELECT COUNT(*) AS rows_where_previous_row_is_not_last_year
    FROM lagged
    WHERE prev_year IS NOT NULL AND prev_year <> year - 1
""").first()[0]

print(f"Rows where LAG would silently skip a year: {gaps:,}")

That's a few hundred rows that would have been mislabelled. The fix is one condition: `prev_year = year - 1`.

Now build the real table. It's saved as `q1_yoy`, with these rules:
- consecutive years only (`prev_year = year - 1`)
- at least `MIN_SALES` sales in **both** years
- `yoy_pct = (median / prev_median − 1) × 100`

This is written in Python with `spark.sql(f"…")`, so `MIN_SALES` comes from the variable in Setup. Change it in one place and the whole analysis follows. The query uses a **CTE** (`WITH lagged AS (…)`), a named sub-query you can read top to bottom, like Power Query steps.

✏️ **TODO 3a:** the consecutive-year condition.
✏️ **TODO 3b:** the year-on-year percentage, rounded to 1 decimal place.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("q1_yoy")} AS
WITH lagged AS (
    SELECT *,
           LAG(year)            OVER w AS prev_year,
           LAG(sales)           OVER w AS prev_sales,
           LAG(median_aed_sqft) OVER w AS prev_median_aed_sqft,
           LAG(off_plan_pct)    OVER w AS prev_off_plan_pct
    FROM {T("agg_price_community_yearly")}
    WINDOW w AS (PARTITION BY community, segment, bedroom_band ORDER BY year)
)
SELECT year, is_full_year, community, segment, bedroom_band, bedroom_sort,
       prev_sales, sales,
       prev_median_aed_sqft, median_aed_sqft,
       prev_off_plan_pct, off_plan_pct,
       ROUND(off_plan_pct - prev_off_plan_pct, 1)               AS mix_shift_pts,
       ABS(off_plan_pct - prev_off_plan_pct) >= {MIN_SHIFT_PTS} AS is_mix_shift,
       ___ AS yoy_pct                                  -- TODO 3b
FROM lagged
WHERE ___                                              -- TODO 3a
  AND prev_sales >= {MIN_SALES}
  AND sales      >= {MIN_SALES}
""")

yoy = spark.table(T("q1_yoy"))
check_close("Rows in q1_yoy", yoy.count(), 1578)
check_close("Qualifying groups in 2025 (vs 2024)", yoy.filter("year = 2025").count(), 250)

## Step 4: `ROW_NUMBER` vs `RANK` vs `DENSE_RANK`, on a toy example

🧠 **Why we're doing this**
These three differ **only when there are ties**, and interviewers love asking about exactly that. Run this toy query (`VALUES` builds a tiny table inline) and compare the three columns where B and C tie.

In [ ]:
%sql
SELECT name, score,
       ROW_NUMBER() OVER (ORDER BY score DESC) AS row_number,
       RANK()       OVER (ORDER BY score DESC) AS rank,
       DENSE_RANK() OVER (ORDER BY score DESC) AS dense_rank
FROM VALUES ('A', 90), ('B', 85), ('C', 85), ('D', 70) AS t(name, score)

- `ROW_NUMBER`: 1, 2, 3, 4. The tie is broken arbitrarily. Use it when you need *exactly one* row (e.g. the latest record per customer).
- `RANK`: 1, 2, 2, **4**. Ties share a rank, then a number is skipped (like a sports league table).
- `DENSE_RANK`: 1, 2, 2, **3**. Ties share, with no gap.

For "top 10 risers" we use `RANK`: if two communities tie for 10th, both deserve to be shown.

## Step 5: Top 10 risers and fallers in 2025

🧠 **Why we're doing this**
Now the headline: **which groups rose most in 2025 compared with 2024?** We rank *within each segment* (`PARTITION BY segment`), because apartments and villas are different markets, and a single list would mix them.

**`QUALIFY`** filters on a window function's result, the way `HAVING` filters on an aggregate. Without it you'd need a sub-query (window functions can't go in `WHERE`, because `WHERE` runs *before* they're calculated). It's supported by Databricks, Snowflake and BigQuery, but not PostgreSQL, so mention that if asked.

✏️ **TODO 5:** complete the `RANK()` window, partitioned by `segment` and ordered by `yoy_pct` **descending**, and the `QUALIFY` to keep ranks 1 to 10.

In [ ]:
%sql
-- TODO 5: top 10 risers per segment, 2025 vs 2024
SELECT segment,
       RANK() OVER (___) AS riser_rank,
       community, bedroom_band,
       prev_median_aed_sqft AS median_2024,
       median_aed_sqft      AS median_2025,
       yoy_pct,
       prev_off_plan_pct    AS off_plan_pct_2024,
       off_plan_pct         AS off_plan_pct_2025,
       is_mix_shift,
       prev_sales           AS sales_2024,
       sales                AS sales_2025
FROM workspace.dubai_re.q1_yoy
WHERE year = 2025
QUALIFY ___
ORDER BY segment, riser_rank

In [ ]:
top_apartment = spark.sql(f"""
    SELECT community, bedroom_band, yoy_pct FROM {T("q1_yoy")}
    WHERE year = 2025 AND segment = 'Apartment'
    ORDER BY yoy_pct DESC LIMIT 1
""").first()
print(f"Fastest-rising apartment group in 2025: {top_apartment['community']} ({top_apartment['bedroom_band']}), {top_apartment['yoy_pct']:+.1f}%")
check_close("Its year-on-year change (%)", top_apartment["yoy_pct"], 97.5, tolerance=0.10)

✏️ **TODO 5b (your turn, no template):** write the **fallers** query, the 10 biggest *declines* per segment in 2025. Copy the risers query into a new `%sql` cell and change only what's needed. *(Two small edits. The solution notebook has it if you get stuck.)*

In [ ]:
%sql
-- TODO 5b: top 10 fallers per segment, 2025 vs 2024
___

## Step 5c: Is the top riser real? The mix effect

🧠 **Why we're doing this**
Look at the top of your apartment risers: a group nearly **doubling** its price per sq ft in one year. Before putting that in a README, ask the analyst's question: **are the same kind of homes selling at higher prices, or are different homes selling?**

A median price per sq ft moves for two reasons:
1. **Price change:** the same kind of homes sell for more.
2. **Mix change:** the *mix* of homes sold changes. In Dubai the biggest mix driver is **off-plan vs ready**, because new off-plan launches usually price well above older ready stock in the same community.

So if a community's sales swing from mostly ready to mostly off-plan, its median jumps even if no individual home went up in value. Statisticians call this a **composition effect**. It's one of the most common ways headline numbers mislead.

That's why `q1_yoy` carries `off_plan_pct` for both years and flags `is_mix_shift` when the share moved by `MIN_SHIFT_PTS` (25) points or more. Run this to see the top riser split by market:

In [ ]:
%sql
-- The #1 apartment riser, split into ready vs off-plan
SELECT d.year, f.market,
       COUNT(*)                        AS sales,
       ROUND(MEDIAN(f.price_per_sqft)) AS median_aed_sqft
FROM workspace.dubai_re.fact_sales f
JOIN workspace.dubai_re.dim_date     d ON f.date_key     = d.date_key
JOIN workspace.dubai_re.dim_location l ON f.location_key = l.location_key
JOIN workspace.dubai_re.dim_property p ON f.property_key = p.property_key
WHERE l.community = 'Motor City' AND p.segment = 'Apartment' AND p.bedroom_band = '2 BR'
  AND d.year IN (2024, 2025) AND NOT f.is_price_outlier
GROUP BY d.year, f.market
ORDER BY f.market, d.year

**Read it like this:** within *ready* homes, the median rose by roughly a quarter. Off-plan barely moved. But off-plan went from about a quarter of sales to about two thirds, and off-plan sells at around twice the price per sq ft. Most of the headline "+97%" is **mix, not appreciation**.

✏️ **TODO 5c:** re-run the risers list **without** the mix-shift groups. Add one condition to the `WHERE`.

In [ ]:
%sql
-- TODO 5c: top 10 apartment risers in 2025, excluding mix shifts
SELECT RANK() OVER (ORDER BY yoy_pct DESC) AS riser_rank,
       community, bedroom_band, prev_median_aed_sqft AS median_2024, median_aed_sqft AS median_2025,
       yoy_pct, off_plan_pct AS off_plan_pct_2025, sales AS sales_2025
FROM workspace.dubai_re.q1_yoy
WHERE year = 2025 AND segment = 'Apartment'
  AND ___
QUALIFY riser_rank <= 10
ORDER BY riser_rank

In [ ]:
r = spark.sql(f"""
    SELECT COUNT_IF(is_mix_shift) AS shifted,
           MAX(CASE WHEN segment = 'Apartment' AND NOT is_mix_shift THEN yoy_pct END) AS top_clean
    FROM {T("q1_yoy")} WHERE year = 2025
""").first()
check_close("2025 groups flagged as mix shifts", r["shifted"], 44, tolerance=0.10)
check_close("Top apartment riser after excluding mix shifts (%)", r["top_clean"], 64.4, tolerance=0.10)

💡 **The honest limit.** Even with a steady off-plan share, the *projects* being sold change from year to year (a new premium tower launches, an older one sells out). The gold standard is a **repeat-sales index**: the same unit sold twice, compared with itself. This dataset has no unit ID (DQ13), so that isn't possible here. Say this in your findings. Knowing the limits of your method is a strength in an interview, not a weakness.

## Step 6: Market breadth, how widespread was the rise?

🧠 **Why we're doing this**
A top-10 list shows the extremes. **Breadth** shows the *typical* experience: across all qualifying groups, what share rose, and by how much (median)? Analysts and investors read breadth as the health of the market. Strong breadth means the rise is everywhere; weak breadth means a few hotspots are pulling the averages up.

✏️ **TODO 6:** complete the share of groups that rose: `AVG(CASE WHEN yoy_pct > 0 THEN 1.0 ELSE 0 END) * 100`.
*Hint:* averaging 1s and 0s gives a proportion. It's the same trick as `AVG(CAST(is_off_plan AS INT))`.

In [ ]:
%sql
-- TODO 6: breadth by year and segment
SELECT year, segment,
       COUNT(*)                                AS groups,
       ROUND(___, 1)                           AS pct_groups_rising,
       ROUND(MEDIAN(yoy_pct), 1)               AS median_yoy_pct
FROM workspace.dubai_re.q1_yoy
WHERE year >= 2019
GROUP BY year, segment
ORDER BY segment, year

In [ ]:
b25 = spark.sql(f"""
    SELECT AVG(CASE WHEN yoy_pct > 0 THEN 1.0 ELSE 0 END) * 100 AS pct_rising, MEDIAN(yoy_pct) AS med
    FROM {T("q1_yoy")} WHERE year = 2025 AND segment = 'Apartment'
""").first()
check_close("2025 apartments: % of groups rising", float(b25["pct_rising"]), 87.1, tolerance=0.05)
check_close("2025 apartments: median YoY %", float(b25["med"]), 10.4, tolerance=0.15)

💡 Click **+ → Visualization → Line** on the breadth result: `year` on X, `median_yoy_pct` on Y, grouped by `segment`. Look at 2020 vs 2022, and how 2025 compares with the 2022–2023 peak.

## Step 7: The longer view, growth from 2021 to 2025 (CAGR)

🧠 **Why we're doing this**
One year can be noisy. Investors and developers care about **growth through the whole post-COVID cycle**. We compare **2021** (the year the market turned) with **2025** (the last full year).

**Pivoting with conditional aggregation.** We need 2021 and 2025 side by side *in one row*. `MAX(CASE WHEN year = 2021 THEN median_aed_sqft END)` picks the 2021 value out of each group's rows. It's the SQL version of an Excel PivotTable with years as columns.

**CAGR (compound annual growth rate)** turns a 4-year change into an equivalent steady yearly rate:

```
CAGR = (end / start) ^ (1 / years) − 1
```

A 60% rise over 4 years is a CAGR of about 12.5% a year, not 15%, because growth compounds. In SQL: `POWER(end / start, 1.0 / 4) - 1`. (Same as `=(B2/A2)^(1/4)-1` in Excel.)

✏️ **TODO 7:** the `cagr_pct` expression, rounded to 1 decimal place.

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {T("q1_growth_2021_2025")} AS
WITH pivoted AS (
    SELECT community, segment, bedroom_band, bedroom_sort,
           MAX(CASE WHEN year = 2021 THEN median_aed_sqft END) AS median_2021,
           MAX(CASE WHEN year = 2025 THEN median_aed_sqft END) AS median_2025,
           MAX(CASE WHEN year = 2021 THEN sales END)           AS sales_2021,
           MAX(CASE WHEN year = 2025 THEN sales END)           AS sales_2025
    FROM {T("agg_price_community_yearly")}
    WHERE year IN (2021, 2025)
    GROUP BY community, segment, bedroom_band, bedroom_sort
)
SELECT *,
       ROUND((median_2025 / median_2021 - 1) * 100, 1)  AS total_growth_pct,
       ___                                             AS cagr_pct           -- TODO 7
FROM pivoted
WHERE sales_2021 >= {MIN_SALES} AND sales_2025 >= {MIN_SALES}
""")

g = spark.table(T("q1_growth_2021_2025"))
check_close("Groups with enough sales in both 2021 and 2025", g.count(), 142)
display(g.orderBy(F.desc("cagr_pct")).limit(10))

🤔 **Think about it:**
1. Why are there far fewer groups here than in the 2025 year-on-year list?
2. A group with no sales at all in 2021 (e.g. a community launched in 2023) gets NULL for `median_2021`. What happens to it, and is that the right outcome for *this* question?

## Step 8: Is the growth still going? 2026 like-for-like momentum

🧠 **Why we're doing this**
The profile showed sales activity cooling sharply through 2026. Are *prices* cooling too? Comparing **2026 year-to-date** with **full-year 2025** would be unfair: different months, different seasons. A **like-for-like** comparison uses *the same months* in both years: January to the last complete month of 2026, against January to that same month of 2025.

The last complete month is read from `dim_date`, not typed in, so it updates with new data.

We go back to the **star schema** (fact + dimensions), because the yearly aggregate can't be split by month. Then we join the result to the 2025 year-on-year table, to label each group:

| 2025 vs 2024 | 2026 YTD vs 2025 YTD | Label |
|--------------|----------------------|-------|
| up | up | Still rising |
| up | down | Turning down |
| down | up | Recovering |
| down | down | Still falling |

✏️ **TODO 8:** complete the `CASE` with the two missing labels.

In [ ]:
LAST_MONTH = spark.sql(f"""
    SELECT MAX(month) FROM {T("dim_date")}
    WHERE is_complete_month AND year = (SELECT MAX(year) FROM {T("dim_date")} WHERE is_complete_month)
""").first()[0]
print(f"Like-for-like window: January to month {LAST_MONTH} of 2025 and 2026")

spark.sql(f"""
CREATE OR REPLACE TABLE {T("q1_momentum_2026")} AS
WITH ytd AS (
    SELECT l.community, p.segment, p.bedroom_band, p.bedroom_sort,
           COUNT(CASE WHEN d.year = 2025 THEN 1 END)                   AS sales_ytd_2025,
           COUNT(CASE WHEN d.year = 2026 THEN 1 END)                   AS sales_ytd_2026,
           MEDIAN(CASE WHEN d.year = 2025 THEN f.price_per_sqft END)   AS median_ytd_2025,
           MEDIAN(CASE WHEN d.year = 2026 THEN f.price_per_sqft END)   AS median_ytd_2026
    FROM {T("fact_sales")} f
    JOIN {T("dim_date")}     d ON f.date_key     = d.date_key
    JOIN {T("dim_location")} l ON f.location_key = l.location_key
    JOIN {T("dim_property")} p ON f.property_key = p.property_key
    WHERE d.year IN (2025, 2026)
      AND d.month <= {LAST_MONTH}
      AND NOT f.is_price_outlier
      AND p.segment IN ('Apartment', 'Villa / Townhouse')
    GROUP BY l.community, p.segment, p.bedroom_band, p.bedroom_sort
),
momentum AS (
    SELECT *, ROUND((median_ytd_2026 / median_ytd_2025 - 1) * 100, 1) AS ytd_change_pct
    FROM ytd
    WHERE sales_ytd_2025 >= {MIN_SALES} AND sales_ytd_2026 >= {MIN_SALES}
)
SELECT m.*, y.yoy_pct AS yoy_2025_pct,
       CASE WHEN y.yoy_pct > 0 AND m.ytd_change_pct > 0 THEN 'Still rising'
            WHEN y.yoy_pct > 0 AND m.ytd_change_pct <= 0 THEN ___          -- TODO 8
            WHEN y.yoy_pct <= 0 AND m.ytd_change_pct > 0 THEN ___          -- TODO 8
            WHEN y.yoy_pct <= 0 AND m.ytd_change_pct <= 0 THEN 'Still falling'
       END AS trend
FROM momentum m
LEFT JOIN {T("q1_yoy")} y
       ON  y.year = 2025
       AND y.community = m.community AND y.segment = m.segment AND y.bedroom_band = m.bedroom_band
""")

mo = spark.table(T("q1_momentum_2026"))
check("Last complete month", LAST_MONTH, 10)
check_close("Groups with enough sales in both YTD windows", mo.count(), 221)
display(mo.groupBy("segment", "trend").count().orderBy("segment", "trend"))

🤔 **Think about it:** some rows have `trend = NULL`. Why? (Hint: look at the `LEFT JOIN` and remember the minimum-sales rule in `q1_yoy`.)

In [ ]:
%sql
-- The biggest "turning down" groups by 2025 volume: where did a strong 2025 give way to falling prices?
SELECT community, segment, bedroom_band, yoy_2025_pct, ytd_change_pct, sales_ytd_2025, sales_ytd_2026
FROM workspace.dubai_re.q1_momentum_2026
WHERE trend = 'Turning down'
ORDER BY sales_ytd_2025 DESC
LIMIT 15

## Step 9: Export the results

Same pattern as Module 3: small tables → pandas → CSV in the `exports` volume. Download them into `data/gold/` in your repo. Power BI and the README will use them.

In [ ]:
EXPORT_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/exports/"

for table in ["q1_yoy", "q1_growth_2021_2025", "q1_momentum_2026"]:
    pdf = spark.table(T(table)).toPandas()
    pdf.to_csv(EXPORT_PATH + f"{table}.csv", index=False)
    print(f"✅ {table}.csv  ({len(pdf):,} rows)")

## Step 10: Write up your findings

🧠 **Why we're doing this**
Recruiters spend about 30 seconds on a README. They don't read your code first. They read **what you found**. A good finding has four parts:

1. **What:** the observation, with the number. *"JVC 1-bed apartments, the single busiest group (about 9,100 sales), rose about 13.5% in 2025 (AED 1,249 → 1,418 per sq ft)…"*
2. **Compared with what:** the benchmark. *"…slightly above the 10.4% median across ~200 qualifying apartment groups."*
3. **So what:** why a developer, broker or buyer should care.
4. **Caveat:** the limit of the evidence. *"Medians of 50+ sales; outliers excluded; price per sq ft does not adjust for building age or quality."*

✏️ **TODO 10:** open `docs/findings/q1_price_hotspots.md` in VS Code and fill in the blanks **from your own results above**. Then send it to me and I'll review it like a hiring manager would.

Avoid: *"prices went up a lot"*, numbers without a comparison, headline risers that are really mix shifts (Step 5c), and claims about *why* (the data shows *what* happened, not why).

## ✅ Module 4 complete

**What you built:** three results tables answering Q1 at three time horizons (2025 year-on-year, 2021–2025 CAGR, 2026 like-for-like momentum), all with a stated minimum-sample rule, plus your written findings.

**🎤 Practise saying these out loud** (model answers in `LEARNING.md`):
1. What's the difference between `GROUP BY` and a window function?
2. `ROW_NUMBER` vs `RANK` vs `DENSE_RANK`: when would you use each?
3. What's the "gap trap" with `LAG`, and how did you handle it?
3b. Your top riser nearly doubled in a year. Is that real? (The mix effect.)
4. Why can't you filter on a window function in `WHERE`? What did you use instead?
5. Why a minimum of 50 sales, and what did it cost you?
6. What is CAGR, and why not just divide the total growth by 4?
7. What is a like-for-like comparison, and why did you need one for 2026?

**Commit your work**
1. **File → Export → IPython Notebook**, then save over `databricks/04_q1_price_hotspots.ipynb`
2. Download the three CSVs into `data/gold/`
3. In VS Code:
```bash
git add databricks/04_q1_price_hotspots.ipynb data/gold/ docs/findings/q1_price_hotspots.md
git commit -m "Add Q1 price hotspots: YoY, 2021-2025 CAGR, 2026 momentum, findings"
git push
```
4. Tick Module 4 in the README roadmap.

**Next:** Module 5, Q2 off-plan vs ready in **Python** (VS Code): hypothesis testing and bootstrap confidence intervals.